In [ ]:
!pip install shapely
!pip install geovoronoi
!pip install numpy==1.22.0 #конкретная версия numpy, так как после обновления с другими версиями появляются ошибки
!pip install osmnx # подключение OSM
!pip install geojson
!pip install geopandas
!pip install networkx
!pip install matplotlib==3.1.3 # импортирую конкретную версию, тк есть ошибки при построении графиков
!pip install folium

In [ ]:
from shapely.geometry import LineString, Polygon, Point
import geojson
from geojson import Feature, FeatureCollection, dump
import pyproj
import pandas as pd
import geopandas as gpd
from scipy import spatial
import json
import networkx as nx
import numpy as np
import matplotlib 
import matplotlib.pyplot as plt
import osmnx as ox
import os
from scipy.spatial import ConvexHull, convex_hull_plot_2d

In [ ]:
os.chdir("C:/...")

Задачи, решаемые, при выполнении этой лабораторной работы:
1.	Рассчитать зоны действия точек интереса, используя хабы;
2.	Рассчитать зоны действия точек интереса, используя полигоны Вороного;
3.	Рассчитать изохроны для 2 (или более) точек интереса изохроны (5, 10, 15, 20 мин), используя API адрес;
4.	Рассчитать изохроны для 2 (или более) точек интереса изохроны (5, 10, 15, 20 мин), используя рассчитанные пешеходные маршруты;
5.	Подготовить данные о наземном общественном транспорте в выбранной области исследования.


Финальные картинки:


1.   Зоны действия на основе хабов вокруг точек притяжения (3 категории, значит, 3 графика)
2. Полигоны вороного
3. Изохроны, построенные 2 способами
4. 2 тепловые карты: взвешенная и стандартная
5. Карта наземного общественного транспорта


**Изохроны на основе Выпуклых оболочек и пешеходных расстояний**

In [ ]:
d = gpd.read_file('population.geojson')

In [ ]:
# функция рассчёта в расстояния 
def dist(p0, p1):  
    a, b = Point(p0), Point(p1)
    dist = a.distance(b)
    return dist

In [ ]:
# функция поиска ближайшей точки
def search_nearest_point(Tree_Graph, point):
    sub = Tree_Graph.query([point], 1)   # запрос к графу
    id_node = sub[1].tolist()[0]     # выбор ближайшей точки
    closest_node = (Tree_Graph.data[id_node][0], Tree_Graph.data[id_node][1])
    return closest_node

In [ ]:
# функция расчёта кратчайшего пути
def path(P, Tree_Graph, Hub):
    closest_Hub = search_nearest_point(Tree_Graph, Hub)
    Paths = {}
    for point in P:
        closest_point = search_nearest_point(Tree_Graph, point)
        path = nx.dijkstra_path(G, closest_point, closest_Hub, weight = 'weight')
        distance = 0
        for i in range(len(path)-1):
            distance += dist(path[i], path[i+1])
        Paths[point] = {'path': path, 'distance': distance}
    return Paths

In [ ]:
# функция, реализующая алгоритм Джарвиса
def rotate(A,B,C):
    return (B[0] - A[0])*(C[1]- B[1]) - (B[1] - A[1])*(C[0] - B[0])
def jarvismarch(A):
    n = len(A)
    P = list(range(n))
    # start point
    for i in range(1, n):
        if A[P[i]][0] < A[P[0]][0]:
            P[i], P[0] = P[0], P[i]
    H = [P[0]]
    del P[0]
    P.append(H[0])
    while True:
        right = 0
        for i in range(1, len(P)):
            if rotate(A[H[-1]], A[P[right]], A[P[i]]) < 0:
                right = i
        if P[right] == H[0]:
            break
        else:
            H.append(P[right])
            del P[right]
    return H     # алгоритм возвращает последовательность точек для выпуклой оболочки

In [ ]:
# словарь с переменными, которые в дальнейшем будут заполняться центроидами услуг из геодатафремов
a = {'houses': [], 'place_of_worship': [], 'post_office': [], 'pharmacy': []}   
for c in a:
    b = gpd.read_file(c + ".geojson")   # открываю файлы, загруженные в colab
    globals()['gdf_'+ c] = gpd.GeoDataFrame(b)   # превращаю в gdf 
    globals()['gdf_'+ c].to_crs(4326)            # устанавливаю систему координат Псевдомеркатор
    for p in globals()['gdf_'+ c]['geometry']:   # в каждом gdf обращаюсь построчно к столбцу "geometry"
        a[c].append(p.centroid)   # определяю координаты центроида полигона, который изначально указан в столбце "geomerty"

In [ ]:
minX, minY, maxX, maxY = 30.209141, 59.945586, 30.335134, 59.982592 

In [ ]:
with open('pedestrian_roads.geojson', encoding = 'utf-8', errors='ignore') as json_file:
    data = json.load(json_file)['features']

In [ ]:
G = nx.DiGraph()

for edge in data:
    coord= edge['geometry']['coordinates']
    distance = edge['properties']['distance']
    G.add_edge((coord[0][0], coord[0][1]), (coord[1][0], coord[1][1]), weight = distance)
del edge, coord, distance, data

In [ ]:
Tree_Graph = spatial.KDTree(list(G.nodes()))   # KDTree из графа дорог
Hub = (30.294412, 59.959957)     # ст. м. Петроградская (30.311421, 59.966444) и Питстанция Лавашово (30.294412, 59.959957) 

In [ ]:
# координаты жилых зданий, которые в радиусе 1300 метров от точки притяжения
P = []
for point in list(d['geometry']):
    if dist(point, Hub) <= 1300:
        P.append((point.x, point.y))

In [ ]:
import time
start = time.time()
Paths = path(P, Tree_Graph, Hub)     # рассчитываем все кратчайшие пути и расстояния
speed = 110                           #  (м/мин) - приблизительная скорость ходьбы человека. потом исправить на 85

end = (time.time() - start) / 60
print(end)

In [ ]:
limits = {20: [], 15: [], 10: [], 5: []}    # собираем точки, которые попадают в указанные лимиты
for point in Paths.keys():
    walk = Paths[point]['distance']*100000
    time_walk = walk/speed
    for time in limits.keys():
        if time_walk <= time:
            limits[time].append(point)

In [ ]:
isoh = []
for time, points in limits.items():
    H = jarvismarch(points)
    polygon = [limits[time][index_point] for index_point in H]
    isoh.append([time, Polygon(polygon)])

In [ ]:
pop1 = {20: [], 15: [], 10: [], 5: []}    # собираем точки, которые попадают в указанные лимиты=
for i in range(4):
    for j in range(len(d)):
        if d['geometry'][j].within(isoh[i][1]) == True:
            pop1[list(pop1.keys())[i]].append(d['Количество проживающих'][j])

In [ ]:
popul1 = [sum(pop1[list(pop1.keys())[i]]) for i in range(len(pop1.keys()))]

In [ ]:
heat = gpd.read_file('population.geojson')
heat

In [ ]:
heat['lon'] = ''
heat['lat'] = ''
heat['lon'] = d.apply(lambda x: x.geometry.x, axis = 1)
heat['lat'] = d.apply(lambda x: x.geometry.y, axis = 1)

In [ ]:
Heat = gpd.GeoDataFrame(heat)

In [ ]:
F = gpd.GeoDataFrame(index = [c[0] for c in isoh], crs='epsg:4326', geometry = [c[1] for c in isoh])

In [ ]:
F['Изохроны доступности кафе Питстанция Лавашово (лучшая шаверма в СПб), минуты'] = [20, 15, 10, 5]

In [ ]:
F.explore(tiles = 'CartoDB positron', 
          column = 'Изохроны доступности кафе Питстанция Лавашово (лучшая шаверма в СПб), минуты')

## Через Кеплер

In [ ]:
import keplergl
from keplergl import KeplerGl

In [ ]:
map = KeplerGl(height = 4000)
map.add_data(F.copy(), name = 'f')
map.add_data(heat.copy(), name = 'heat')

In [ ]:
map

## Изохроны с использованием Mapbox API

In [ ]:
import requests 

point = (30.294412, 59.959957)  
key = 'your key'
transit = '/walking/'
times = [5, 10, 15, 20]

url = ''.join(['https://api.mapbox.com/isochrone/v1/mapbox', 
               transit, str(point[0]), ',', str(point[1]), '?', 
              'contours_minutes=5,10,15,20', '&',
              'contours_colors=6706ce,04e813,4286f4,F9E784', '&',
               'polygons=true', '&', 
               'access_token=your key',             
              ])

req = requests.get(url)

features = req.json()['features']   # json с данными о маршруте

In [ ]:
isochrones = [Polygon(c['geometry']['coordinates'][0]) for c in features]

In [ ]:
pop = {20: [], 15: [], 10: [], 5: []}    # собираем точки, которые попадают в указанные лимиты=
for i in range(4):
    for j in range(len(d)):
        if d['geometry'][j].within(isochrones[i]) == True:
            pop[list(pop.keys())[i]].append(d['Количество проживающих'][j])

In [ ]:
popul = [sum(pop[list(pop.keys())[i]]) for i in range(len(pop.keys()))]
# isochrones.append(Point((30.311421, 59.966444)))

In [ ]:
popul

In [ ]:
popul1    # через пешеходные расстояния

In [ ]:
F_map = gpd.GeoDataFrame(index = range(len(isochrones)), crs='epsg:4326', geometry = isochrones)

In [ ]:
F_map['Потенциальный пешеходный поток, чел.'] = popul
F_map['MapBox Изохроны доступности до ст.м. Петроградская, минуты'] = [20, 15, 10, 5]

In [ ]:
F_map.explore(tiles = 'CartoDB positron', 
          column = 'MapBox Изохроны доступности до ст.м. Петроградская, минуты',
             popup = True, tooltip = 'Потенциальный пешеходный поток, чел.')

# mo.explore(
#      column="ADMIN_L5", # make choropleth based on "BoroName" column
#      tooltip="NAME", # show "BoroName" value in tooltip (on hover)
#      popup=True, # show all values in popup (on click)
#      tiles="CartoDB positron", # use "CartoDB positron" tiles
#      cmap="Set1", # use "Set1" matplotlib colormap
#      style_kwds=dict(color="black") # use black outline
#     )

In [ ]:
map = KeplerGl(height = 4000)
map.add_data(F.copy(), name = 'pedestrian')
map.add_data(heat.copy(), name = 'heat')
map.add_data(F_map.copy(), name = 'Mapbox')

In [ ]:
map

Диаграммы Вороного

In [ ]:
!pip install geovoronoi

In [ ]:
import geovoronoi
from geovoronoi import voronoi_regions_from_coords, points_to_coords

In [ ]:
minX, minY, maxX, maxY = 30.209141, 59.945586, 30.335134, 59.982592 

In [ ]:
boundary_shape = Polygon([(minX, minY), (minX, maxY), (maxX, maxY), (maxX, minY)])

In [ ]:
poly_shapes, poly_to_pt_assignments = voronoi_regions_from_coords(a['pharmacy'], boundary_shape)

In [ ]:
Pha = gpd.GeoDataFrame(index = list(poly_shapes.keys()), crs='epsg:4326', geometry = list(poly_shapes.values()))

In [ ]:
poly_shapes

In [ ]:
Pha.explore()

## Выпуклые оболочки вокруг хабов

In [ ]:
from geovoronoi import voronoi_regions_from_coords, points_to_coords

In [ ]:
area_name = 'Петроградский район, Санкт-Петербург'
area = ox.geocode_to_gdf(area_name)   # информацию из OSM по моему запросу превращает в геодф 
area.explore(tiles = 'CartoDB positron')  # воспроизводит интерактивную карту

In [ ]:
# через Полигоны вороного
# poly_shapes, poly_to_pt_assignments = voronoi_regions_from_coords(a['pharmacy'], area['geometry'][0]) 
# Pha = gpd.GeoDataFrame(index = list(poly_shapes.keys()), crs='epsg:4326',
#                        geometry = [a['pharmacy'][i] for i in range(len(a['pharmacy']))])
# Pha1 = gpd.GeoDataFrame(index = list(poly_shapes.keys()), crs='epsg:4326',
#                        geometry = [list(poly_shapes.values())[i] for i in range(len(a['pharmacy']))])
# Pha2 = pd.concat([Pha, Pha1])
# Pha2.set_crs('EPSG:4326')

In [ ]:
Pha2.explore(tiles = 'CartoDB positron')

In [ ]:
houses = gpd.GeoDataFrame(index = range(len(a['houses'])), crs='epsg:4326',
                       geometry = [c for c in a['houses']])

In [ ]:
# словарь с наименьшими расстояниями по каждому виду услуг от центроидов домов
b = {'place_of_worship': [], 'post_office': [], 'pharmacy': []} 
for centr in houses['geometry']: # перебор по центроидам домов
    for c in b:   # перебор по элементам словаря
        min_dist = 1000000000   # максимально возможное расстояние для определения наименьшего в будущем
        for point in a[c]:   # перебор поэлементно внутри списков центроидов точек притяжения
            dist1 = dist(centr, point)   # определение расстояния между 2 точками
            if dist1 < min_dist:
                min_dist = dist1
                coord=(point.x, point.y)
        b[c].append(coord)

In [ ]:
houses['place_of_worship'] = b['place_of_worship']
houses['post_office'] = b['post_office']
houses['pharmacy'] = b['pharmacy']

In [ ]:
hulls = []
for i in houses['place_of_worship'].unique():
    coord = [(c.x, c.y) for c in houses[houses['place_of_worship']==i]['geometry']]
    if len(coord) >= 3:    # для построения выпуклой диаграммы не менее 3 точек 
        hull = ConvexHull(coord)
        coords = [coord[c] for c in hull.vertices]    
        # достаю координаты точек по индексам граничных точек (результат функции .vertices)
        hulls.append(Point(i))
        hulls.append(Polygon(coords))

In [ ]:
hull_gdf = gpd.GeoDataFrame(index = range(len(hulls)), crs='epsg:4326',
                         geometry = hulls)

In [ ]:
hull_gdf.explore(
     tiles="CartoDB positron", # use "CartoDB positron" tiles
#      cmap="Set1", # use "Set1" matplotlib colormap
     style_kwds=dict(color = 'Green', fillColor = 'blue', fillOpacity = 0.1),
#      legend_kwds=dict(caption = 'Выпуклые оболочки для категории Почта') 
            )

## Теплокарты

In [ ]:
pop = pd.read_excel('Население_районы.xlsx')

In [ ]:
ar = ox.geocode_to_gdf(list(pop['Район']))

In [ ]:
area = gpd.GeoDataFrame(ar)

In [ ]:
area['population'] = pop['Население']
area['weight'] = pop['Взвешенное количество населения']

In [ ]:
area

In [ ]:
area.explore(column="population", # make choropleth based on "BoroName" column
             tooltip="population", # show "BoroName" value in tooltip (on hover)
             popup=True, # show all values in popup (on click)
             tiles="CartoDB positron", # use "CartoDB positron" tiles
             cmap="Set1", # use "Set1" matplotlib colormap
             style_kwds=dict(color="black")) # use black outline


In [ ]:
import keplergl
from keplergl import KeplerGl

In [ ]:
map = KeplerGl(height = 500)
map.add_data(area)

In [ ]:
map